# Experimentos de pontos pré-jogo
Comparamos sete variantes na mesma amostra. A linha de avaliação é simulada a partir da média pré-jogo do jogador. As probabilidades medem a chance de superar essa linha.

Poisson e binomial negativa usam a mesma média da projeção ponderada. Os modelos de contexto ajustam a média em etapas: minutos dos 5 jogos anteriores, ritmo estimado das equipes e defesa adversária. Todos os indicadores usam datas anteriores à previsão.

In [ ]:
from pathlib import Path
import sys
import pandas as pd

root = Path.cwd()
if root.name == 'notebooks':
    root = root.parent
sys.path.insert(0, str(root))

from nba_props_backtest import summarize
from src.data.history import download_archive_season
from src.features.context import CONTEXT_COLUMNS
from src.models.experiments import EXPERIMENT_LABELS, experiment_chart, run_experiments

# Preencha com os dois arquivos Parquet para executar offline.
local_files = []
seasons = ('2024-25', '2025-26')
revision = '7339613e45d1bfcb4c98276e764a8b68bea04e30'
inputs = local_files or [download_archive_season(season, revision) for season in seasons]
games = pd.concat([pd.read_parquet(item, columns=CONTEXT_COLUMNS) for item in inputs], ignore_index=True)
games.shape

In [ ]:
predictions = run_experiments(games)
metrics, calibration = summarize(predictions, EXPERIMENT_LABELS)
metrics.round(4)

In [ ]:
pd.Series({
    'jogadores': predictions.player_id.nunique(),
    'primeira previsão': predictions.date.min(),
    'última previsão': predictions.date.max(),
    'fração de fallback Poisson na binomial negativa': predictions.nb_used_poisson.mean(),
})

In [ ]:
experiment_chart(metrics, calibration).show()

## Limitações
O ritmo e a defesa são estimativas por box score. Os minutos projetados usam o histórico recente de partidas jogadas. O benchmark tem limiar simulado e o contexto usa regras fixas para esta amostra. Para medir generalização, repita a análise em outro período e reserve esse período para a avaliação.